In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [ ]:
pip install plotly

In [ ]:
df = pd.read_csv("IMDb Movies India.csv", encoding='latin1')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df['Year'] = df['Year'].astype(str)

df['Year'] = df['Year'].str.extract('(\d+)')

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

In [ ]:
df['Duration'] = df['Duration'].astype(str)

df['Duration'] = df['Duration'].str.replace(' min', '')

df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')

In [ ]:
df['Votes'] = df['Votes'].astype(str)

df['Votes'] = df['Votes'].str.replace(',', '')

df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')

In [ ]:
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

In [ ]:
df.isnull().sum()

In [ ]:
df['Genre'].fillna('Unknown', inplace=True)
df['Director'].fillna('Unknown', inplace=True)

df['Actor 1'].fillna('Unknown', inplace=True)
df['Actor 2'].fillna('Unknown', inplace=True)
df['Actor 3'].fillna('Unknown', inplace=True)

In [ ]:
df.dropna(inplace=True)

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(df['Rating'], bins=20, kde=True)

plt.title("Movie Rating Distribution")

plt.show()

In [ ]:
top_genres = df['Genre'].value_counts().head(10)

plt.figure(figsize=(10,6))

sns.barplot(x=top_genres.values, y=top_genres.index)

plt.title("Top 10 Genres")

plt.show()

In [ ]:
top_directors = df['Director'].value_counts().head(10)

plt.figure(figsize=(10,6))

sns.barplot(x=top_directors.values, y=top_directors.index)

plt.title("Top Directors")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(x='Votes', y='Rating', data=df)

plt.title("Votes vs Ratings")

plt.show()

In [ ]:
numeric_df = df[['Year', 'Duration', 'Votes', 'Rating']]

plt.figure(figsize=(8,5))

sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
df['Movie_Age'] = 2026 - df['Year']

In [ ]:
df['Genre_Count'] = df['Genre'].apply(lambda x: len(str(x).split(',')))

In [ ]:
director_counts = df['Director'].value_counts()

df['Director_Frequency'] = df['Director'].map(director_counts)

In [ ]:
actor_counts = df['Actor 1'].value_counts()

df['Actor1_Frequency'] = df['Actor 1'].map(actor_counts)

In [ ]:
le_genre = LabelEncoder()
le_director = LabelEncoder()

df['Genre_Encoded'] = le_genre.fit_transform(df['Genre'])

df['Director_Encoded'] = le_director.fit_transform(df['Director'])

In [ ]:
features = [
    'Year',
    'Duration',
    'Votes',
    'Movie_Age',
    'Genre_Count',
    'Director_Frequency',
    'Actor1_Frequency',
    'Genre_Encoded',
    'Director_Encoded'
]

X = df[features]

y = df['Rating']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, predictions)

print("MAE:", mae)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("RMSE:", rmse)

In [ ]:
r2 = r2_score(y_test, predictions)

print("R² Score:", r2)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, predictions)

plt.xlabel("Actual Ratings")

plt.ylabel("Predicted Ratings")

plt.title("Actual vs Predicted Ratings")

plt.show()

In [ ]:
joblib.dump(model, "movie_rating_predictor.pkl")

In [ ]:
df['combined_features'] = (
    df['Genre'].astype(str) + ' ' +
    df['Director'].astype(str) + ' ' +
    df['Actor 1'].astype(str) + ' ' +
    df['Actor 2'].astype(str)
)

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [ ]:
def recommend_movies(movie_name):

    movie_name = movie_name.lower()

    indices = pd.Series(df.index, index=df['Name'].str.lower()).drop_duplicates()

    if movie_name not in indices:
        return "Movie not found"

    idx = indices[movie_name]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:6]

    movie_indices = [i[0] for i in similarity_scores]

    return df['Name'].iloc[movie_indices]

In [ ]:
recommend_movies("3 Idiots")

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
fig = px.scatter(
    df,
    x='Votes',
    y='Rating',
    color='Genre',
    hover_data=['Name'],
    title='Interactive Votes vs Rating'
)

fig.show()

In [ ]:
top_movies = df.sort_values(by='Rating', ascending=False).head(10)

top_movies[['Name', 'Rating']]

In [ ]:
importance = model.feature_importances_

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importance
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)

plt.title("Feature Importance")

plt.show()

In [ ]:
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

joblib.dump(cosine_sim, "cosine_similarity.pkl")

In [ ]:
def predict_movie_rating(
    year,
    duration,
    votes,
    genre_count,
    director_frequency,
    actor_frequency,
    genre_encoded,
    director_encoded
):

    movie_age = 2026 - year

    input_data = pd.DataFrame([[
        year,
        duration,
        votes,
        movie_age,
        genre_count,
        director_frequency,
        actor_frequency,
        genre_encoded,
        director_encoded
    ]], columns=features)

    prediction = model.predict(input_data)

    return prediction[0]

In [ ]:
predict_movie_rating(
    year=2020,
    duration=120,
    votes=5000,
    genre_count=2,
    director_frequency=10,
    actor_frequency=15,
    genre_encoded=5,
    director_encoded=20
)

In [ ]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

print("Linear Regression R2 Score:",
      r2_score(y_test, lr_predictions))

In [ ]:
print("Random Forest R2:",
      r2_score(y_test, predictions))

In [ ]:
top_movies = df.sort_values(
    by='Rating',
    ascending=False
).head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    x='Rating',
    y='Name',
    data=top_movies,
    palette='viridis'
)

plt.title("Top 10 Highest Rated Movies")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(df['Votes'], bins=30, kde=True)

plt.xscale('log')

plt.title("Votes Distribution")

plt.show()

In [ ]:
df.to_csv("cleaned_imdb_movies.csv", index=False)

In [ ]:
!pip install streamlit